In [1]:
import numpy as np
import pandas as pd

In [2]:
RAW_PATH = '../Data/ALL.csv'
OUT_PATH = "cars_clean.csv"

In [3]:
# 0) Load the raw data
df = pd.read_csv(RAW_PATH)
report = {"raw_rows": len(df)}

In [4]:
# 1) Drop rows that are completely empty
df = df.dropna(how="all").copy()
report["after_drop_empty_rows"] = len(df)

In [5]:
# 1b) Keep USED cars only (the recommender will only recommend used cars)
df = df[df["condition"] == "Used"].copy()
report["after_keep_used_only"] = len(df)

In [6]:
# 2) Drop useless columns
#    - currency / country : a single value in every row (EGP / Egypt) -> zero information
#    - trim: ~84% missing and messy text ("A/T / Highline", "Automatic / Baseline"...)
#    - bodyType: only filled for Toyota (~89% missing)
#    - region : ~45% missing and mixes governorates (Cairo) with districts (Maadi);
#         `city` already gives the location in a cleaner way

In [7]:
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print("Constant columns:", constant_cols)
df = df.drop(columns=constant_cols + ["trim", "title", "bodyType", "region"])
df = df.rename(columns={"price": "price_egp"})

Constant columns: ['currency', 'country', 'condition']


In [8]:
# 3) Fix data types
for col in ["id", "year", "km", "price_egp"]:
    df[col] = df[col].astype("int64")

In [9]:
# 4) Standardise text columns
#A) remove the make prefix so the same model is not written in 2 ways
def strip_make_prefix(row):
    model, make = row["model"], row["make"]
    if model.lower().startswith(make.lower() + " "):
        return model[len(make) + 1:]
    return model

df["model"] = df.apply(strip_make_prefix, axis=1)

In [10]:
# 4b) "Other Hyundai" / "Other Kia" / "Other Nissan" all mean "unknown model"
df["model"] = df["model"].replace({"Other Hyundai": "Other",
                                   "Other Kia": "Other",
                                   "Other Nissan": "Other"})

In [11]:
df["fuel_type"] = df["fuelType"].replace({"Gas": "Petrol", "Natural Gas": "CNG"})
df = df.drop(columns="fuelType")

df = df.rename(columns={"sellerType": "seller_type"})

In [12]:
# 5) Fix impossible / suspicious `km` values
bad_km = (df["km"] > 1_000_000) | (df["km"] == 0)
df["km"] = df["km"].astype("float64")      # float is needed to hold NaN
df.loc[bad_km, "km"] = np.nan
report["km_set_to_nan"] = int(bad_km.sum())

In [13]:
# 6) Remove duplicate listings
cols_without_id = [c for c in df.columns if c != "id"]
dups = df.duplicated(subset=cols_without_id, keep="first")
report["duplicates_removed"] = int(dups.sum())
df = df[~dups].copy()

In [14]:
# 7) Memory-friendly dtypes for low-cardinality text columns
# ------------------------------------------------------------------
for col in ["make", "fuel_type", "transmission", "seller_type"]:
    df[col] = df[col].astype("category")

In [15]:
report["final_rows"] = len(df)
df.to_csv(OUT_PATH, index=False)

In [16]:
# === Cleaning report ===
for k, v in report.items():
    print(f"{k:28s} {v}")
print("\nMissing values after cleaning:\n", df.isna().sum()[lambda s: s > 0])

raw_rows                     3620
after_drop_empty_rows        3617
after_keep_used_only         3296
km_set_to_nan                185
duplicates_removed           31
final_rows                   3265

Missing values after cleaning:
 km       184
color     58
dtype: int64
